In [1]:
# Celda 1
%cd C:/Users/fabi_/Documents/workflow/textos-catolicos-xix

import os
os.environ["OMP_THREAD_LIMIT"] = "1"  # One thread per Tesseract process; parallelism comes from WORKERS below
import csv
import re
import shutil
import statistics
import tempfile
import time
from concurrent.futures import ThreadPoolExecutor
import psutil  # Reads the free memory
import pymupdf  # Turns each PDF page into an image
import pytesseract
from PIL import Image

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# To test on the pilot sample (5 pages per book), use instead:
# PDF_FOLDER = "ocr-piloto/muestra" and OCR_FOLDER = "ocr-piloto/textos-ocr-muestra"
# The corpus lives on the external drive. Each PDF is copied to C: before it is read (cell 5):
PDF_FOLDER = "E:/Datos/textos-catolicos-xix/biblioteca_religiosa"
OCR_FOLDER = "textos-ocr"  # One CSV of lines per book
TEMP_FOLDER = os.path.join(tempfile.gettempdir(), "ocr-corpus")  # On C:, in case the external drive disconnects
TESSERACT_MODEL = "spa"
DPI = 300
WORKERS = 3  # Pages read at the same time (the PC has 8 cores and 8 GB of RAM). Pilot: 0.7-1.1 s per page with 6, 2.5 s with 1
MIN_FREE_GB = 1.0  # Before each group of pages, wait while the free memory is below this
BATCH_MINUTES = 60  # Each run stops after about an hour (at the end of the book in progress); run again to go on

C:\Users\fabi_\Documents\workflow\textos-catolicos-xix


In [2]:
# Celda 2
# List the PDFs (the folder also has one EPUB, which is skipped):
pdf_files = []
for file_name in sorted(os.listdir(PDF_FOLDER)):
    if file_name.endswith(".pdf"):
        pdf_files.append(file_name)
os.makedirs(OCR_FOLDER, exist_ok=True)
os.makedirs(TEMP_FOLDER, exist_ok=True)
print(len(pdf_files), "PDFs")

239 PDFs


In [3]:
# Celda 3
# OCR of one page image. Tesseract returns one row per word, with its box in pixels and the id of its line:
def read_words(image):
    data = pytesseract.image_to_data(image, lang=TESSERACT_MODEL, output_type=pytesseract.Output.DICT)
    words = []
    for k in range(len(data["text"])):
        text = data["text"][k].strip()
        if text == "":
            continue
        words.append({"text": text,
                      "line": (data["block_num"][k], data["par_num"][k], data["line_num"][k]),  # Tesseract's line id
                      "left": data["left"][k], "top": data["top"][k],
                      "right": data["left"][k] + data["width"][k], "bottom": data["top"][k] + data["height"][k],
                      "height": data["height"][k]})
    return words

# Words grouped into lines, keeping each line's box and the median height of its words.
# Lines keep Tesseract's reading order (left column before right column).
# The cleaning notebook (5-limpieza) uses the boxes and heights to find notes, headers and small type:
def group_lines(words):
    lines = {}
    for word in words:
        lines.setdefault(word["line"], []).append(word)
    rows = []
    for line_words in lines.values():
        rows.append({"left": min(word["left"] for word in line_words),
                     "top": min(word["top"] for word in line_words),
                     "right": max(word["right"] for word in line_words),
                     "bottom": max(word["bottom"] for word in line_words),
                     "height": statistics.median(word["height"] for word in line_words),
                     "text": " ".join(word["text"] for word in line_words)})
    return rows

In [4]:
# Celda 4
# Two columns read as one. When the gutter is narrow (e.g. Rodríguez, Ejercicio, 1861, with a rule between columns),
# Tesseract sometimes reads straight across both columns: "tuo, et ex tota anima tua, et ez|san Agustin , nos".
# Pilot: 34 of 476 pages of that book. Such a page has many long lines crossing the middle of the page:
def crossing_lines(rows, page_width):
    count = 0
    for row in rows:
        if row["left"] < 0.35 * page_width and row["right"] > 0.65 * page_width and len(row["text"]) > 40:
            count = count + 1
    return count

# On other pages only a few lines are read across, but the rule shows up inside them: "tan aje-|terés, sino por":
def rule_inside(rows, page_width):
    for row in rows:
        if row["left"] < 0.35 * page_width and row["right"] > 0.65 * page_width and re.search(r"\w\s*\|\s*\w", row["text"]):
            return True
    return False

# The gutter: the x, between 35% and 65% of the page width, covered by the fewest words.
# The rule, read as "|", does not count; at equal cover the x nearest the middle wins:
def find_gutter(words, page_width):
    cover = [0] * page_width
    for word in words:
        if re.search(r"[^\W\d_]", word["text"]) and "|" not in word["text"]:
            for x in range(word["left"], word["right"]):
                cover[x] = cover[x] + 1
    candidates = range(int(page_width * 0.35), int(page_width * 0.65))
    gutter = min(candidates, key=lambda x: (cover[x], abs(x - page_width // 2)))
    return gutter, cover[gutter]

# Each word goes to the column its center falls in, and the lines are rebuilt column by column.
# No second OCR is needed: the words are the same, only their grouping changes.
# A word with the rule inside ("aprove-|logos") is split in two; the rule alone ("|") is dropped:
def split_columns(words, gutter):
    left = []
    right = []
    for word in words:
        if "|" in word["text"]:
            before, after = word["text"].split("|", 1)
            after = after.replace("|", "")
            if before != "":
                left.append(dict(word, text=before, right=gutter - 1, line=word["line"] + ("left",)))
            if after != "":
                right.append(dict(word, text=after, left=gutter + 1, line=word["line"] + ("right",)))
        elif (word["left"] + word["right"]) / 2 < gutter:
            left.append(dict(word, line=word["line"] + ("left",)))
        else:
            right.append(dict(word, line=word["line"] + ("right",)))
    return group_lines(left) + group_lines(right)

def read_page(image):
    words = read_words(image)
    rows = group_lines(words)
    if crossing_lines(rows, image.width) >= 5 or rule_inside(rows, image.width):
        # A single-column page also has long lines crossing the middle, but no blank strip runs down it:
        # every x is covered by words in most lines (pilot: over half of them).
        # Two columns read as one leave a strip few words cover (pilot: 1 to 8 words, on pages of 41 to 80 lines):
        gutter, cover = find_gutter(words, image.width)
        if cover <= 0.2 * len(rows):
            rows = split_columns(words, gutter)
    return rows

In [5]:
# Celda 5
# OCR every page of every PDF, one CSV per book, e.g. "agreda-...-1860.pdf" -> "agreda-...-1860.csv".
# Each PDF is copied to C:, read, and its copy deleted.
# Pages are rendered in batches, so a 900-page book never sits in memory all at once.
# A book whose CSV already exists is skipped, so the cell can be stopped and run again;
# the CSV is written under a temporary name and renamed at the end, so a stopped book starts over.
# The corpus (about 58,000 pages) takes some 16-17 hours, so it runs in batches of about BATCH_MINUTES:
COLUMNS = ["page", "page_width", "page_height", "left", "top", "right", "bottom", "height", "text"]

batch_start = time.time()
for file_name in pdf_files:
    output_path = os.path.join(OCR_FOLDER, file_name[:-4] + ".csv")
    if os.path.exists(output_path):
        continue
    if time.time() - batch_start > BATCH_MINUTES * 60:
        print("Batch done. Run this cell again to go on.")
        break
    local_copy = os.path.join(TEMP_FOLDER, file_name)
    shutil.copy(os.path.join(PDF_FOLDER, file_name), local_copy)
    book = pymupdf.open(local_copy)
    with open(output_path + ".part", "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, COLUMNS)
        writer.writeheader()
        with ThreadPoolExecutor(WORKERS) as pool:
            for start in range(0, len(book), WORKERS * 2):
                while psutil.virtual_memory().available < MIN_FREE_GB * 1e9:
                    time.sleep(30)  # Other programs are using the memory: wait instead of running out of it
                images = []
                for page in book.pages(start, min(start + WORKERS * 2, len(book))):
                    pixmap = page.get_pixmap(dpi=DPI, colorspace=pymupdf.csGRAY)
                    images.append(Image.frombytes("L", [pixmap.width, pixmap.height], pixmap.samples))
                for k, rows in enumerate(pool.map(read_page, images)):
                    for row in rows:
                        row["page"] = start + k + 1  # Page of the PDF, from 1
                        row["page_width"] = images[k].width
                        row["page_height"] = images[k].height
                        writer.writerow(row)
    page_count = len(book)
    book.close()
    os.remove(local_copy)
    os.replace(output_path + ".part", output_path)
    print(file_name, page_count, "pages,", round((time.time() - batch_start) / 60), "minutes into the batch")

done = 0
for file_name in pdf_files:
    if os.path.exists(os.path.join(OCR_FOLDER, file_name[:-4] + ".csv")):
        done = done + 1
print(done, "of", len(pdf_files), "books done")

pinamonti-breves-meditaciones-sobre-los-novisimos.pdf 42 pages, 1 minutes into the batch


pinamonti-breves-meditaciones-sobre-los-novisimos-zaragoza-1853.pdf 52 pages, 1 minutes into the batch


rosignoli-verdades-eternas.pdf 426 pages, 12 minutes into the batch
239 of 239 books done
